In [1]:
import csv
import pandas as pd
class fragmenter:
    # tested with Python 3.8.8 and RDKit version 2021.09.4
    
    from rdkit import Chem
    import marshal as marshal
    from rdkit.Chem import rdmolops

    # does a substructure match and then checks whether the match 
    # is adjacent to previous matches
    @classmethod
    def get_substruct_matches(cls, mol_searched_for, mol_searched_in, atomIdxs_to_which_new_matches_have_to_be_adjacent):
        
        valid_matches = []
        
        if mol_searched_in.GetNumAtoms() >= mol_searched_for.GetNumAtoms():
            matches = mol_searched_in.GetSubstructMatches(mol_searched_for)
            
            if matches:
                for match in matches:
                        add_this_match = True
                        if len(atomIdxs_to_which_new_matches_have_to_be_adjacent) > 0:
                            add_this_match = False
                            
                            for i in match:
                                for neighbor in mol_searched_in.GetAtomWithIdx(i).GetNeighbors():
                                    if neighbor.GetIdx() in atomIdxs_to_which_new_matches_have_to_be_adjacent:
                                        add_this_match = True
                                        break
                                
                        if add_this_match:
                            valid_matches.append(match)
                
        return valid_matches
    
    # count heavier isotopes of hydrogen correctly
    @classmethod
    def get_heavy_atom_count(cls, mol):
        heavy_atom_count = 0
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() != 1:
                heavy_atom_count += 1
        
        return heavy_atom_count
    
    def __init__(self, fragmentation_scheme = {}, fragmentation_scheme_order = None, match_hydrogens = False, algorithm = '', n_atoms_cuttoff = -1, function_to_choose_fragmentation = False, n_max_fragmentations_to_find = -1):

        if not type(fragmentation_scheme) is dict:
            raise TypeError('fragmentation_scheme must be a dctionary with integers as keys and either strings or list of strings as values.')
            
        if len(fragmentation_scheme) == 0:
            raise ValueError('fragmentation_scheme must be provided.')
        
        if not algorithm in ['simple', 'complete', 'combined']:
            raise ValueError('Algorithm must be either simple ,complete or combined.')
            
        if algorithm == 'simple':
            if n_max_fragmentations_to_find != -1:
                raise ValueError('Setting n_max_fragmentations_to_find only makes sense with complete or combined algorithm.')
        
        self.algorithm = algorithm
        
        if algorithm in ['combined', 'complete']:
            if n_atoms_cuttoff == -1:
                raise ValueError('n_atoms_cuttoff needs to be specified for complete or combined algorithms.')
                
            if function_to_choose_fragmentation == False:
                raise ValueError('function_to_choose_fragmentation needs to be specified for complete or combined algorithms.')
                
            if not callable(function_to_choose_fragmentation):
                raise TypeError('function_to_choose_fragmentation needs to be a function.')
            else:
                if type(function_to_choose_fragmentation([{}, {}])) != dict:
                    raise TypeError('function_to_choose_fragmentation needs to take a list of fragmentations and choose one of it')
                
            if n_max_fragmentations_to_find != -1:
                if n_max_fragmentations_to_find < 1:
                    raise ValueError('n_max_fragmentations_to_find has to be 1 or higher.')

        if fragmentation_scheme_order is None:
            fragmentation_scheme_order = []

        if algorithm in ['simple', 'combined']:
            assert len(fragmentation_scheme) == len(fragmentation_scheme_order)
        else:
            fragmentation_scheme_order = [key for key in fragmentation_scheme.keys()]
            
        self.n_max_fragmentations_to_find = n_max_fragmentations_to_find
        
        self.n_atoms_cuttoff = n_atoms_cuttoff

        self.match_hydrogens = match_hydrogens
        
        self.fragmentation_scheme = fragmentation_scheme
        
        self.function_to_choose_fragmentation = function_to_choose_fragmentation
        
        # create a lookup dictionaries to faster finding a group number
        self._fragmentation_scheme_group_number_lookup = {}
        self._fragmentation_scheme_pattern_lookup = {}
        self.fragmentation_scheme_order = fragmentation_scheme_order

        for group_number, list_SMARTS in fragmentation_scheme.items():
            
            if type(list_SMARTS) is not list:
                list_SMARTS = [list_SMARTS]
                
            for SMARTS in list_SMARTS:
                if SMARTS != '':
                    self._fragmentation_scheme_group_number_lookup[SMARTS] = group_number
                    
                    mol_SMARTS = fragmenter.Chem.MolFromSmarts(SMARTS)
                    self._fragmentation_scheme_pattern_lookup[SMARTS] = mol_SMARTS

    def fragment(self, SMILES_or_molecule):
        
        if type(SMILES_or_molecule) is str:
            mol_SMILES = fragmenter.Chem.MolFromSmiles(SMILES_or_molecule)
            mol_SMILES = fragmenter.Chem.AddHs(mol_SMILES) if self.match_hydrogens else mol_SMILES
            is_valid_SMILES = mol_SMILES is not None
            
            if not is_valid_SMILES:
                raise ValueError('Following SMILES is not valid: ' + SMILES_or_molecule)

        else:
            mol_SMILES = SMILES_or_molecule

        # iterate over all separated molecules
        success = []
        fragmentation = {}
        fragmentation_matches = {}
        for mol in fragmenter.rdmolops.GetMolFrags(mol_SMILES, asMols = True):

            this_mol_fragmentation, this_mol_success = self.__get_fragmentation(mol)
    
            for SMARTS, matches in this_mol_fragmentation.items():
                group_number = self._fragmentation_scheme_group_number_lookup[SMARTS]
                
                if not group_number in fragmentation:
                    fragmentation[group_number] = 0
                    fragmentation_matches[group_number] = []
                
                fragmentation[group_number] += len(matches)
                fragmentation_matches[group_number].extend(matches)   
                
            success.append(this_mol_success)

        return fragmentation, all(success), fragmentation_matches
    
    def fragment_complete(self, SMILES_or_molecule):
        
        if type(SMILES_or_molecule) is str:
            mol_SMILES = fragmenter.Chem.MolFromSmiles(SMILES_or_molecule)
            mol_SMILES = fragmenter.Chem.AddHs(mol_SMILES) if self.match_hydrogens else mol_SMILES
            is_valid_SMILES = mol_SMILES is not None
            
            if not is_valid_SMILES:
                raise ValueError('Following SMILES is not valid: ' + SMILES_or_molecule)

        else:
            mol_SMILES = SMILES_or_molecule

        if len(fragmenter.rdmolops.GetMolFrags(mol_SMILES)) != 1:
            raise ValueError('fragment_complete does not accept multifragment molecules.')

        temp_fragmentations, success = self.__complete_fragmentation(mol_SMILES)

        fragmentations = []
        fragmentations_matches = []
        for temp_fragmentation in temp_fragmentations:
            fragmentation = {}
            fragmentation_matches = {}
            for SMARTS, matches in temp_fragmentation.items():
                group_number = self._fragmentation_scheme_group_number_lookup[SMARTS]

                fragmentation[group_number] = len(matches)
                fragmentation_matches[group_number] = matches

            fragmentations.append(fragmentation)
            fragmentations_matches.append(fragmentation_matches)

        return fragmentations, success, fragmentations_matches


    def __get_fragmentation(self, mol_SMILES):
        
        success = False
        fragmentation = {}
        if self.algorithm in ['simple', 'combined']:
            fragmentation, success = self.__simple_fragmentation(mol_SMILES)
        
        if success:
            return fragmentation, success
        
        if self.algorithm in ['combined', 'complete']:
            fragmentations, success = self.__complete_fragmentation(mol_SMILES)
            
            if success:
                fragmentation = self.function_to_choose_fragmentation(fragmentations)
        
        return fragmentation, success
    
    def __simple_fragmentation(self, mol_SMILES):

        if self.match_hydrogens:
            target_atom_count = len(mol_SMILES.GetAtoms())
        else:
            target_atom_count = fragmenter.get_heavy_atom_count(mol_SMILES)
    
        success = False
        fragmentation = {}
        
        fragmentation, atomIdxs_included_in_fragmentation = self.__search_non_overlapping_solution(mol_SMILES, {}, set(), set())
        success = len(atomIdxs_included_in_fragmentation) == target_atom_count
        
        # if not successful, clean up molecule and search again
        level = 1
        while not success:
            fragmentation_so_far , atomIdxs_included_in_fragmentation_so_far = fragmenter.__clean_molecule_surrounding_unmatched_atoms(mol_SMILES, fragmentation, atomIdxs_included_in_fragmentation, level)
            level += 1
            
            if len(atomIdxs_included_in_fragmentation_so_far) == 0:
                break
            
            fragmentation_so_far, atomIdxs_included_in_fragmentation_so_far = self.__search_non_overlapping_solution(mol_SMILES, fragmentation_so_far, atomIdxs_included_in_fragmentation_so_far, atomIdxs_included_in_fragmentation_so_far)
            
            success = len(atomIdxs_included_in_fragmentation_so_far) == target_atom_count
            
            if success:
                fragmentation = fragmentation_so_far
            
        return fragmentation, success
    
    def __search_non_overlapping_solution(self, mol_searched_in, fragmentation, atomIdxs_included_in_fragmentation, atomIdxs_to_which_new_matches_have_to_be_adjacent):
        
        n_atomIdxs_included_in_fragmentation = len(atomIdxs_included_in_fragmentation) - 1
        
        while n_atomIdxs_included_in_fragmentation != len(atomIdxs_included_in_fragmentation):
            n_atomIdxs_included_in_fragmentation = len(atomIdxs_included_in_fragmentation)
            
             
            for group_number in self.fragmentation_scheme_order:
                list_SMARTS = self.fragmentation_scheme[group_number]
                if type(list_SMARTS) is not list:
                    list_SMARTS = [list_SMARTS]
                
                for SMARTS in list_SMARTS:
                    if SMARTS != "":  
                        fragmentation, atomIdxs_included_in_fragmentation = self.__get_next_non_overlapping_match(mol_searched_in, SMARTS, fragmentation, atomIdxs_included_in_fragmentation, atomIdxs_to_which_new_matches_have_to_be_adjacent)

        return fragmentation, atomIdxs_included_in_fragmentation

    def __get_next_non_overlapping_match(self, mol_searched_in, SMARTS, fragmentation, atomIdxs_included_in_fragmentation, atomIdxs_to_which_new_matches_have_to_be_adjacent):
        
        mol_searched_for = self._fragmentation_scheme_pattern_lookup[SMARTS]
        
        if atomIdxs_to_which_new_matches_have_to_be_adjacent:
            matches = fragmenter.get_substruct_matches(mol_searched_for, mol_searched_in, atomIdxs_to_which_new_matches_have_to_be_adjacent)
        else:
            matches = fragmenter.get_substruct_matches(mol_searched_for, mol_searched_in, set())
        
        if matches:
            for match in matches:
                all_atoms_of_new_match_are_unassigned = atomIdxs_included_in_fragmentation.isdisjoint(match)
        
                if all_atoms_of_new_match_are_unassigned:
                    if not SMARTS in fragmentation:
                        fragmentation[SMARTS] = []
                        
                    fragmentation[SMARTS].append(match)
                    atomIdxs_included_in_fragmentation.update(match)   
                
        return fragmentation, atomIdxs_included_in_fragmentation

    @classmethod
    def __clean_molecule_surrounding_unmatched_atoms(cls, mol_searched_in, fragmentation, atomIdxs_included_in_fragmentation, level):
    
        for i in range(0, level):
            
            atoms_missing = set(range(0, fragmenter.get_heavy_atom_count(mol_searched_in))).difference(atomIdxs_included_in_fragmentation)
                        
            new_fragmentation = fragmenter.marshal.loads(fragmenter.marshal.dumps(fragmentation))
            
            for atomIdx in atoms_missing:
                for neighbor in mol_searched_in.GetAtomWithIdx(atomIdx).GetNeighbors():
                    for smart, atoms_found in fragmentation.items():
                        for atoms in atoms_found:
                            if neighbor.GetIdx() in atoms:
                                if smart in new_fragmentation:
                                    if new_fragmentation[smart].count(atoms) > 0:
                                        new_fragmentation[smart].remove(atoms)
                                
                        if smart in new_fragmentation:
                            if len(new_fragmentation[smart]) == 0:
                                new_fragmentation.pop(smart)
                                
                                
            new_atomIdxs_included_in_fragmentation = set()
            for i in new_fragmentation.values():
                for j in i:
                    new_atomIdxs_included_in_fragmentation.update(j)
                    
            atomIdxs_included_in_fragmentation = new_atomIdxs_included_in_fragmentation
            fragmentation = new_fragmentation
            
        return fragmentation, atomIdxs_included_in_fragmentation
       
    def __complete_fragmentation(self, mol_SMILES):
    
        heavy_atom_count = fragmenter.get_heavy_atom_count(mol_SMILES)
        
        if heavy_atom_count > self.n_atoms_cuttoff:
            return {}, False
        
        completed_fragmentations = []
        groups_leading_to_incomplete_fragmentations = []        
        completed_fragmentations, groups_leading_to_incomplete_fragmentations, incomplete_fragmentation_found = self.__get_next_non_overlapping_adjacent_match_recursively(mol_SMILES, heavy_atom_count, completed_fragmentations, groups_leading_to_incomplete_fragmentations, {}, set(), set(), self.n_max_fragmentations_to_find)
        success = len(completed_fragmentations) > 0
        
        return completed_fragmentations, success
        
    def __get_next_non_overlapping_adjacent_match_recursively(self, mol_searched_in, heavy_atom_count, completed_fragmentations, groups_leading_to_incomplete_fragmentations, fragmentation_so_far, atomIdxs_included_in_fragmentation_so_far, atomIdxs_to_which_new_matches_have_to_be_adjacent, n_max_fragmentations_to_find = -1):
      
        n_completed_fragmentations = len(completed_fragmentations)
        incomplete_fragmentation_found = False
        complete_fragmentation_found = False
        
        if len(completed_fragmentations) == n_max_fragmentations_to_find:
            return completed_fragmentations, groups_leading_to_incomplete_fragmentations, incomplete_fragmentation_found
                    
                
        for group_number in self.fragmentation_scheme_order:
            list_SMARTS = self.fragmentation_scheme[group_number]
            
            if complete_fragmentation_found:
                break
            
            if type(list_SMARTS) is not list:
                list_SMARTS = [list_SMARTS]
            
            for SMARTS in list_SMARTS:
                if complete_fragmentation_found:
                    break
                
                if SMARTS != "":  
                    matches = fragmenter.get_substruct_matches(self._fragmentation_scheme_pattern_lookup[SMARTS], mol_searched_in, atomIdxs_included_in_fragmentation_so_far)
                    
                    for match in matches:
                        
                        # only allow non-overlapping matches
                        all_atoms_are_unassigned = atomIdxs_included_in_fragmentation_so_far.isdisjoint(match)
                        if not all_atoms_are_unassigned:
                            continue
                        
                        # only allow matches that do not contain groups leading to incomplete matches
                        for groups_leading_to_incomplete_fragmentation in groups_leading_to_incomplete_fragmentations:
                            if fragmenter.__is_fragmentation_subset_of_other_fragmentation(groups_leading_to_incomplete_fragmentation, fragmentation_so_far):
                                return completed_fragmentations, groups_leading_to_incomplete_fragmentations, incomplete_fragmentation_found
                        
                        # only allow matches that will lead to new fragmentations
                        use_this_match = True
                        n_found_groups = len(fragmentation_so_far)
                        
                        for completed_fragmentation in completed_fragmentations:
                            
                            if not SMARTS in completed_fragmentation:
                                continue
                            
                            if n_found_groups == 0:
                                use_this_match = not fragmenter.__is_match_contained_in_fragmentation(match, SMARTS, completed_fragmentation)
                            else:
                                if fragmenter.__is_fragmentation_subset_of_other_fragmentation(fragmentation_so_far, completed_fragmentation):
                                    use_this_match = not fragmenter.__is_match_contained_in_fragmentation(match, SMARTS, completed_fragmentation)
                            
                            if not use_this_match:
                                break
                                
                        if not use_this_match:
                            continue
                        
                        # make a deepcopy here, otherwise the variables are modified down the road
                        # marshal is used here because it works faster than copy.deepcopy
                        this_SMARTS_fragmentation_so_far = fragmenter.marshal.loads(fragmenter.marshal.dumps(fragmentation_so_far))
                        this_SMARTS_atomIdxs_included_in_fragmentation_so_far = atomIdxs_included_in_fragmentation_so_far.copy()
                        
                        if not SMARTS in this_SMARTS_fragmentation_so_far:
                            this_SMARTS_fragmentation_so_far[SMARTS] = []
                            
                        this_SMARTS_fragmentation_so_far[SMARTS].append(match)
                        this_SMARTS_atomIdxs_included_in_fragmentation_so_far.update(match)
                        
                        # only allow matches that do not contain groups leading to incomplete matches
                        for groups_leading_to_incomplete_match in groups_leading_to_incomplete_fragmentations:
                            if fragmenter.__is_fragmentation_subset_of_other_fragmentation(groups_leading_to_incomplete_match, this_SMARTS_fragmentation_so_far):
                                use_this_match = False
                                break
                            
                        if not use_this_match:
                            continue
                        
                        # if the complete molecule has not been fragmented, continue to do so
                        if len(this_SMARTS_atomIdxs_included_in_fragmentation_so_far) < heavy_atom_count:
                            completed_fragmentations, groups_leading_to_incomplete_fragmentations, incomplete_fragmentation_found = self.__get_next_non_overlapping_adjacent_match_recursively(mol_searched_in, heavy_atom_count, completed_fragmentations, groups_leading_to_incomplete_fragmentations, this_SMARTS_fragmentation_so_far, this_SMARTS_atomIdxs_included_in_fragmentation_so_far, this_SMARTS_atomIdxs_included_in_fragmentation_so_far, n_max_fragmentations_to_find)
                            break
                        
                        # if the complete molecule has been fragmented, save and return
                        if len(this_SMARTS_atomIdxs_included_in_fragmentation_so_far) == heavy_atom_count:                 
                            completed_fragmentations.append(this_SMARTS_fragmentation_so_far)
                            complete_fragmentation_found = True
                            break
                        
        # if until here no new fragmentation was found check whether an incomplete fragmentation was found
        if n_completed_fragmentations == len(completed_fragmentations):      
            
            if not incomplete_fragmentation_found:
                
                incomplete_matched_groups = {}
                
                if len(atomIdxs_included_in_fragmentation_so_far) > 0:
                    unassignes_atom_idx = set(range(0, heavy_atom_count)).difference(atomIdxs_included_in_fragmentation_so_far)
                    for atom_idx in unassignes_atom_idx:
                        neighbor_atoms_idx = [i.GetIdx() for i in mol_searched_in.GetAtomWithIdx(atom_idx).GetNeighbors()]
                        
                        for neighbor_atom_idx in neighbor_atoms_idx:
                            for found_smarts, found_matches in fragmentation_so_far.items():
                                for found_match in found_matches:
                                    if neighbor_atom_idx in found_match:
                                        if not found_smarts in incomplete_matched_groups:
                                            incomplete_matched_groups[found_smarts] = []
                                            
                                        if found_match not in incomplete_matched_groups[found_smarts]:
                                            incomplete_matched_groups[found_smarts].append(found_match)
                                    
                    is_subset_of_groups_already_found = False
                    indexes_to_remove = []
                    
                    for idx, groups_leading_to_incomplete_match in enumerate(groups_leading_to_incomplete_fragmentations):
                        is_subset_of_groups_already_found = fragmenter.__is_fragmentation_subset_of_other_fragmentation(incomplete_matched_groups, groups_leading_to_incomplete_match)
                        if is_subset_of_groups_already_found:
                            indexes_to_remove.append(idx)
                        
                    for index in sorted(indexes_to_remove, reverse=True):
                        del groups_leading_to_incomplete_fragmentations[index]
                        
                    groups_leading_to_incomplete_fragmentations.append(incomplete_matched_groups)
                    groups_leading_to_incomplete_fragmentations = sorted(groups_leading_to_incomplete_fragmentations, key = len)
                    
                    incomplete_fragmentation_found =  True
    
        return completed_fragmentations, groups_leading_to_incomplete_fragmentations, incomplete_fragmentation_found
    
    @classmethod
    def __is_fragmentation_subset_of_other_fragmentation(cls, fragmentation, other_fragmentation):
        n_found_groups = len(fragmentation)
        n_found_other_groups = len(other_fragmentation)
        
        if n_found_groups == 0:
            return False
            
        if n_found_other_groups < n_found_groups:
            return False
        
        n_found_SMARTS_that_are_subset = 0
        for found_SMARTS, _ in fragmentation.items():
            if found_SMARTS in other_fragmentation:
                found_matches_set = set(frozenset(i) for i in fragmentation[found_SMARTS])
                found_other_matches_set =  set(frozenset(i) for i in other_fragmentation[found_SMARTS])
                
                if found_matches_set.issubset(found_other_matches_set):
                    n_found_SMARTS_that_are_subset += 1
            else:
                return False
            
        return n_found_SMARTS_that_are_subset == n_found_groups
    
    @classmethod
    def __is_match_contained_in_fragmentation(cls, match, SMARTS, fragmentation):
        if not SMARTS in fragmentation:
            return False
            
        found_matches_set = set(frozenset(i) for i in fragmentation[SMARTS])
        match_set = set(match)
        
        return match_set in found_matches_set

if __name__ == '__main__':
    # smiles = ['CCC(C)CCCCC(=O)NC(CCN)C(=O)NC(C(C)O)C(=O)NC(CCN)C(=O)NC1CCNC(=O)C(NC(=O)C(NC(=O)C(NC(=O)C(NC(=O)C(NC(=O)C(NC1=O)CCN)CC2=CC=CC=C2)CC(C)C)CCN)CCN)C(C)O','CC1C(N=C(O1)C2=CC=CC=C2O)C(=O)N(CCCCNC(=O)C3=C(C(=CC=C3)O)O)CCCNC(=O)C4=C(C(=CC=C4)O)O', 'CC(C)O', 'CCCCO', 'CCCO', 'CCO', 'CO']
    with open('Smiles_input_EKUT.csv', 'r') as input_csv:
        reader = csv.reader(input_csv)
        # Skip first line (header)
        next(reader)
        # Create the SMILES dictionary
        # Format: {'BS_XXXX': 'CCC(C)CCCCC...', 'BS_YYYY': 'CCCCC...'}
        smiles = {row[0]: row[1] for row in reader}
    
    fragmentation_scheme = {
        'CH3' : '[CH3;X4]',
        'CH2' : '[CH2;X4]',
        'CH' : '[CH1;X4]',
        'C' : ['[CH0;X4]' , '[CH0;X3]'],
        'CH2=CH' : '[CH2]=[CH]',
        'CH=CH' : '[CH]=[CH]',
        'CH2=C' : ['[CH2]=[C]' , '[CH2]=[c]'],
        'CH=C' : ['[CH]=[CH0]' , '[CH]=[cH0]'],
        'ACH' : '[cH]',
        'AC' : '[cH0]',
        'ACCH3' : '[c][CH3;X4]',
        'ACCH2' : '[c][CH2;X4]',
        'ACCH' : '[c][CH;X4]',
        'OH' : '[OH]',
        'CH3OH' : '[CH3][OH]',
        'H2O' : '[OH2]',
        'ACOH' : '[c][OH]',
        'CH3CO' : '[CH3][CH0]=O',
        'CH2CO' : '[CH2][CH0]=O',
        'CH=O' : '[CH]=O',
        'CH3COO' : '[CH3]C	=O[OH0]',
        'CH2COO' : '[CH2]C	=O[OH0]',
        'HCOO' : '[CH]	=O[OH0]',
        'CH3O' : '[CH3][OH0]',
        'CH2O' : '[CH2][OH0]',
        'CHO' : '[CH][OH0]',
        'THF' : '[CH2;R][OH0]',
        'CH3NH2' : '[CH3][NH2]',
        'CH2NH2' : '[CH2][NH2]',
        'CHNH2' : '[CH][NH2]',
        'CH3NH' : '[CH3][NH]',
        'CH2NH' : '[CH2][NH]',
        'CHNH' : '[CH][NH]',
        'CH3N' :  ['[CH3][N]' , '[CH3][n]'],
        'CH2N' : '[CH2][N]',
        'ACNH2' : '[c][NH2]',
        'C5H5N' : 'n1[cH][cH][cH][cH][cH]1',
        'C5H4N' : ['n1[c][cH][cH][cH][cH]1',
               'n1[cH][c][cH][cH][cH]1',
               'n1[cH][cH][c][cH][cH]1'],
        'C5H3N' : ['n1[c][c][cH][cH][cH]1',
               'n1[c][cH][c][cH][cH]1',
               'n1[c][cH][cH][c][cH]1',
               'n1[c][cH][cH][cH][c]1',
               'n1[cH][c][c][cH][cH]1',
               'n1[cH][c][cH][c][cH]1'],
        'CH3CN' : '[CH3]C#N',
        'CH2CN' : '[CH2]C#N',
        'COOH' : 'C	=O[OH]',
        'HCOOH' : '[CH]	=O[OH]',
        'CH2Cl' : '[CH2]Cl',
        'CHCl' : '[CH]Cl',
        'CCl' : '[CH0]Cl',
        'CH2Cl2' : '[CH2]	ClCl',
        'CHCl2' : '[CH]	ClCl',
        'CCl2' : 'C	ClCl',
        'CHCl3' : '[CH]	Cl	ClCl',
        'CCl3' : 'C	Cl	Cl	Cl',
        'CCl4' : 'C	Cl	Cl	Cl	Cl',
        'ACCl' : '[c]Cl',
        'CH3NO2' : '[CH3][N+]	=O[O-]',
        'CH2NO2' : '[CH2][N+]	=O[O-]',
        'CHNO2' : '[CH][N+]	=O[O-]',
        'ACNO2' : '[c][N+]	=O[O-]',
        'CS2' : 'C	=S=S',
        'CH3SH' : '[CH3][SH]',
        'CH2SH' : '[CH2][SH]',
        'Furfural' : 'O=[CH]c1[cH][cH][cH]o1',
        'DOH' : '[OH][CH2][CH2][OH]',
        'I' : '[IH0]',
        'Br' : '[BrH0]',
        'CH#C' : '[CH]#C',
        'C#C' : 'C#C',
        'DMSO' : '[CH3]S	=O[CH3]',
        'ACRY' : '[CH2]=[CH1][C]#N',
        'C=C' : '[CH0]=[CH0]',
        'ACF' : '[c]F',
        'DMF' : '[CH]	=ON	[CH3][CH3]',
        'CF3' : 'C	F	FF',
        'CF2' : 'C	FF',
        'CF' : '[C]F',
        'COO' : ['[CH0]	=O[OH0]' , '[cH0]	=O[oH0]'],
        'SiH3' : '[SiH3]',
        'SiH2' : '[SiH2]',
        'SiH' : '[SiH]',
        'Si' : '[Si]',
        'SiH2O' : '[SiH2][OH0]',
        'SiHO' : '[SiH][OH0]',
        'SiO' : '[Si][OH0]',
        'CCl3F' : 'C	Cl	Cl	ClF',
        'CCl2F' : 'C	Cl	ClF',
        'HCCl2F' : '[CH]	Cl	ClF',
        'HCClF' : '[CH]	ClF',
        'CClF2' : 'C	Cl	FF',
        'HCClF2' : '[CH]	Cl	FF',
        'CClF3' : 'C	Cl	F	FF',
        'CCl2F2' : 'C	Cl	Cl	FF',
        'CONH2' : 'C	=O[NH2]',
        'CONHCH3' : 'C	=O[NH][CH3]',
        'CONHCH2' : 'C	=O[NH][CH2]',
        'CONCH32' : 'C	=ON	[CH3][CH3]',
        'CONCH3CH2' : 'C	=ON	[CH3][CH2]',
        'CONCH22' : 'C	=ON	[CH2][CH2]',
        'CH3S' : '[CH3]S',
        'CH2S' : '[CH2]S',
        'CHS' : '[CH]S',
        'MORPH' : '[CH2]1[CH2][NH][CH2][CH2]O1',
        'C4H4S' : '[cH]1[cH][s;X2][cH][cH]1',
        'C4H3S' : ['[c]1[cH][s;X2][cH][cH]1',
               '[cH]1[c][s;X2][cH][cH]1'],
        'C4H2S' : ['[c]1[c][s;X2][cH][cH]1',
               '[c]1[cH][s;X2][cH][c]1',
               '[cH]1[c][s;X2][c][cH]1',
               '[cH]1[c][s;X2][cH][c]1'],
        'NCO' : 'N=C=O',
        'CH22SU' : '[CH2]S	=O	=O[CH2]',
        'CH2CHSU' : '[CH2]S	=O	=O[CH]'
    }
    fragmentation_scheme_order1 = ['CH3NH2', 'CH2NH2', 'CHNH2', 'CH3NH', 'CH2NH', 'CHNH', 'CH3N', 'CH2N', 'ACNH2', 'CHO', 'CH2O', 'CH3O', 'CH=O', 'CH2CO', 'CH3CO', 'HCOO', 'CH2COO', 'CH3COO', 'CH3', 'CH2', 'CH', 'C', 'CH2=CH', 'CH=CH', 'CH2=C', 'CH=C', 'ACH', 'AC', 'ACCH3', 'ACCH2', 'ACCH', 'OH', 'CH3OH', 'H2O', 'ACOH', 'THF', 'C5H5N', 'C5H4N', 'C5H3N', 'CH3CN', 'CH2CN', 'COOH', 'HCOOH', 'CH2Cl', 'CHCl', 'CCl', 'CH2Cl2', 'CHCl2', 'CCl2', 'CHCl3', 'CCl3', 'CCl4', 'ACCl', 'CH3NO2', 'CH2NO2', 'CHNO2', 'ACNO2', 'CS2', 'CH3SH', 'CH2SH', 'Furfural', 'DOH', 'I', 'Br', 'CH#C', 'C#C', 'DMSO', 'ACRY', 'C=C', 'ACF', 'DMF', 'CF3', 'CF2', 'CF', 'COO', 'SiH3', 'SiH2', 'SiH', 'Si', 'SiH2O', 'SiHO', 'SiO', 'CCl3F', 'CCl2F', 'HCCl2F', 'HCClF', 'CClF2', 'HCClF2', 'CClF3', 'CCl2F2', 'CONH2', 'CONHCH3', 'CONHCH2', 'CONCH32', 'CONCH3CH2', 'CONCH22', 'CH3S', 'CH2S', 'CHS', 'MORPH', 'C4H4S', 'C4H3S', 'C4H2S', 'NCO', 'CH22SU', 'CH2CHSU' 
    ]
    print('simple algorithm 1')
    frg = fragmenter(fragmentation_scheme, fragmentation_scheme_order=fragmentation_scheme_order1, algorithm='simple')
    result_dict = {}
    # Iterate through the SMILES dictionary items
    contador = 1
    for id, smi in smiles.items():
        # If the SMILES string is a representation of a list, use the first SMILES in said list
        if ',' in smi:
            smi = smi.strip('][').split(', ')[0]
        fragmentation, success, fragmentation_matches = frg.fragment(smi)
        result_dict[smi] = fragmentation
        contador+=1
        print(contador)
        #print(smi, fragmentation)
        #print(id, fragmentation)
    result_df = pd.DataFrame.from_dict(result_dict, orient='index')
    result_df.to_csv('fragmentation_results_EKUT.csv')

simple algorithm 1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
